# Doctor–Patient Conversation → Clinical History → SNOMED CT → FHIR
### Runs in Google Colab · PostgreSQL on Neon · Gemini · evaluated on MTS-Dialog

**Pipeline**
```
Conversation (MTS-Dialog, a typed consultation, or an audio recording)
   ↓  Part 5  Clinical entity extraction (Gemini)  → section, term, present/absent, confidence
   ↓  Part 6  SNOMED CT coding: PostgreSQL keyword search + pgvector meaning search → Gemini picks (RAG)
   ↓  Part 7  FHIR R4 resources: Condition, MedicationStatement, AllergyIntolerance, FamilyMemberHistory, Observation, Procedure
   ↓  Part 8  Evaluation against MTS-Dialog expert notes → the results tables and charts
```

**Research questions answered by Part 8** (from the project brief)
| RQ | Question | Measured by |
|---|---|---|
| RQ1 | How accurate are LLMs at extracting clinical entities? | Fact-level precision, recall, F1 against the expert note; section detection |
| RQ2 | How reliable is automatic SNOMED coding? | Coding coverage + confidence; clinician review sheet |
| RQ3 | Can uncertainty estimation reduce hallucinations? | Hallucination rate per confidence level; effect of a confidence threshold |
| RQ4 | How does performance vary? | All metrics per history section (MTS-Dialog has no specialty labels) |

**Data:** MTS-Dialog (Ben Abacha et al., EACL 2023), CC BY 4.0, 1,701 doctor–patient conversations with expert-written note sections.


---
# Setup
Requires two Colab secrets (🔑 panel, notebook access on): `GEMINI_API_KEY` (Google AI Studio) and `NEON_DATABASE_URL` (a Neon PostgreSQL connection string; pgvector is enabled by the notebook). Then **Runtime → Run all**.

---
# Part 1 — Install and settings

In [ ]:
!pip install -q "psycopg[binary]" google-genai pandas requests "fhir.resources>=8" matplotlib openpyxl

In [ ]:
import os

# ---------- secrets (Colab 🔑 panel; outside Colab: environment variables or typed in) ----------
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    DATABASE_URL = userdata.get("NEON_DATABASE_URL")
except Exception:
    from getpass import getpass
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Gemini API key: ")
    DATABASE_URL = os.environ.get("NEON_DATABASE_URL") or getpass("Neon connection string: ")

# ---------- AI ----------
TEXT_MODEL = "gemini-3.5-flash-lite"     # extraction + code choice. If busy (503): "gemini-flash-lite-latest"
JUDGE_MODEL = "gemini-3.5-flash-lite"    # evaluation judge (Part 8)
EMBEDDING_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
SECONDS_BETWEEN_CALLS = 4                # free tier is rate-limited; raise to 6-10 if you see many "busy" messages

# ---------- terminology ----------
TERMINOLOGY = "demo"                     # "demo" (78 made-up DEMO codes) or "snomed" (real SNOMED CT from NHS TRUD, Part 4B)
TRUD_API_KEY = ""                        # only for "snomed": your TRUD API key (TRUD → My account)
TRUD_ITEM = 1799                         # TRUD item number of the SNOMED CT release (see Part 4B)
TOP_K = 10

# ---------- experiment ----------
TEST_SECTIONS = ["CC", "GENHX", "PASTMEDICALHX", "PASTSURGICAL", "MEDICATIONS", "ALLERGY", "FAM/SOCHX", "ROS"]
PER_SECTION = 3                          # conversations per section (3 → 24 in total). None = all 169 history conversations
RANDOM_SEED = 42
CONFIDENCE_THRESHOLD = 0.7               # RQ3: drop findings the model is less sure about

print("Settings ready. Gemini key:", "found" if GEMINI_API_KEY else "MISSING", "| Neon:", "found" if DATABASE_URL else "MISSING")

---
# Part 2 — Gemini helpers
`ask_ai_json` returns a Python dict. `embed_texts` returns 768 numbers per text.
The `pace()` function keeps us under the free-tier limit, and `retry` waits when Gemini answers *429 / busy*.

In [ ]:
import json, re, time
import requests
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)
_last_call = [0.0]


def pace():
    wait = SECONDS_BETWEEN_CALLS - (time.time() - _last_call[0])
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()


def retry(fn, tries=7):
    delay = 10
    for attempt in range(tries):
        try:
            pace()
            return fn()
        except Exception as e:
            if "PerDay" in str(e):
                raise RuntimeError(f"Daily free quota used up for this model. Switch TEXT_MODEL / JUDGE_MODEL "
                                   f"in Part 1 to another model (see ai.dev/rate-limit), or try tomorrow.") from e
            busy = any(s in str(e) for s in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500"))
            if not busy or attempt == tries - 1:
                raise
            print(f"   Gemini busy, waiting {delay}s ...")
            time.sleep(delay)
            delay = min(delay * 2, 120)


def parse_json(text):
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", (text or "").strip())
    return json.loads(text)


def ask_ai_json(prompt, model=None):
    config = types.GenerateContentConfig(response_mime_type="application/json", temperature=0)
    response = retry(lambda: client.models.generate_content(model=model or TEXT_MODEL, contents=prompt, config=config))
    data = parse_json(response.text)
    if isinstance(data, list):   # lite models sometimes answer with a bare list
        data = {"choices": data, "findings": data}
    return data


def embed_texts(texts, kind="document"):
    vectors = []
    task = "RETRIEVAL_QUERY" if kind == "query" else "RETRIEVAL_DOCUMENT"
    for i in range(0, len(texts), 100):
        batch = texts[i:i + 100]
        config = types.EmbedContentConfig(output_dimensionality=EMBED_DIM, task_type=task)
        result = retry(lambda: client.models.embed_content(model=EMBEDDING_MODEL, contents=batch, config=config))
        vectors += [e.values for e in result.embeddings]
    return vectors

In [ ]:
print(ask_ai_json('In one sentence, what is FHIR in healthcare? Return JSON: {"answer": "..."}'))
print("embedding size:", len(embed_texts(["chest pain"])[0]))

---
# Part 3 — PostgreSQL on Neon
| Table | Holds |
|---|---|
| `concepts` | SNOMED concepts: id, preferred name, semantic tag (disorder, finding, substance …) |
| `descriptions` | Every name/synonym + full-text index + **embedding** (pgvector) |
| `code_maps` | SNOMED → ICD-10 / OPCS-4 |
| `results` | One row per conversation: findings, FHIR bundle, evaluation |

In [ ]:
import psycopg
import pandas as pd

class AutoConnection:
    # Neon's free tier sleeps after ~5 idle minutes and closes connections.
    # This wrapper checks the connection before every use and reconnects if needed.
    def __init__(self, url):
        self.url, self._conn = url, None

    def _live(self):
        if self._conn is not None and not self._conn.closed:
            try:
                self._conn.execute("SELECT 1")
                return self._conn
            except psycopg.OperationalError:
                pass
        print("   (re)connecting to Neon ...")
        self._conn = psycopg.connect(self.url, autocommit=True)
        return self._conn

    def execute(self, *args, **kwargs):
        return self._live().execute(*args, **kwargs)

    def cursor(self, *args, **kwargs):
        return self._live().cursor(*args, **kwargs)

    def transaction(self, *args, **kwargs):
        return self._live().transaction(*args, **kwargs)

    def close(self):
        if self._conn is not None:
            self._conn.close()


conn = AutoConnection(DATABASE_URL)
print(conn.execute("SELECT version()").fetchone()[0][:40], "... connected")


def to_vector(values):
    return "[" + ",".join(f"{float(x):.7g}" for x in values) + "]"


conn.execute(f'''
CREATE EXTENSION IF NOT EXISTS vector;
CREATE TABLE IF NOT EXISTS app_meta (key TEXT PRIMARY KEY, value TEXT);
CREATE TABLE IF NOT EXISTS concepts (
    concept_id TEXT PRIMARY KEY, fsn TEXT, preferred TEXT, semantic_tag TEXT,
    active BOOLEAN DEFAULT TRUE, source TEXT);
CREATE TABLE IF NOT EXISTS descriptions (
    description_id TEXT PRIMARY KEY,
    concept_id TEXT NOT NULL REFERENCES concepts(concept_id) ON DELETE CASCADE,
    term TEXT NOT NULL, is_fsn BOOLEAN DEFAULT FALSE,
    tsv TSVECTOR GENERATED ALWAYS AS (to_tsvector('english', term)) STORED,
    embedding VECTOR({EMBED_DIM}));
CREATE INDEX IF NOT EXISTS descriptions_concept_idx ON descriptions (concept_id);
CREATE INDEX IF NOT EXISTS descriptions_tsv_idx ON descriptions USING GIN (tsv);
CREATE INDEX IF NOT EXISTS descriptions_embedding_hnsw ON descriptions USING hnsw (embedding vector_cosine_ops);
CREATE INDEX IF NOT EXISTS concepts_tag_idx ON concepts (semantic_tag);
CREATE TABLE IF NOT EXISTS code_maps (
    concept_id TEXT NOT NULL, code_system TEXT NOT NULL, code TEXT NOT NULL, code_term TEXT,
    map_group INTEGER DEFAULT 1, map_priority INTEGER DEFAULT 1, map_rule TEXT,
    PRIMARY KEY (concept_id, code_system, code));
CREATE TABLE IF NOT EXISTS results (
    item_id TEXT PRIMARY KEY, source TEXT, section_header TEXT, dialogue TEXT, gold TEXT,
    findings JSONB, fhir JSONB, judge JSONB, settings JSONB, created_at TIMESTAMPTZ DEFAULT now());
''')

stored = conn.execute("SELECT value FROM app_meta WHERE key = 'embedder'").fetchone()
if stored and stored[0] != EMBEDDING_MODEL:
    print(f"⚠️ Stored embeddings came from {stored[0]}; reset them before switching models.")
conn.execute("INSERT INTO app_meta VALUES ('embedder', %s) ON CONFLICT (key) DO UPDATE SET value = EXCLUDED.value",
             (EMBEDDING_MODEL,))
print("Tables ready")

---
# Part 4 — Terminology (the code list)

**4A — Demo (default):** 78 common terms with **made-up IDs (DEMO-001 …)**. Fine to test the pipeline;
coding results on MTS-Dialog will be low because the list is tiny.

**4B — Real SNOMED CT (for real coding results):** register free at **NHS TRUD** (isd.digital.nhs.uk/trud),
subscribe to a SNOMED CT RF2 release (UK Monolith Edition, RF2 Snapshot is the complete one), and find its **item
number** in the page address (`…/items/<number>/releases`). Copy your **API key** from *My account*.
Put both in the Part 1 settings and set `TERMINOLOGY = "snomed"`.
Only clinically useful hierarchies are loaded (disorders, findings, situations, procedures, substances, observables)
so it fits the Neon free storage. Embeddings are created **on demand** while coding, so nothing needs embedding up front.

In [ ]:
DEMO_CONCEPTS = [
    ('DEMO-001', 'Chest pain', 'finding', ['Pain in chest', 'Chest discomfort']),
    ('DEMO-002', 'Chest tightness', 'finding', ['Tight chest', 'Feeling of chest constriction']),
    ('DEMO-003', 'Central chest pain', 'finding', ['Retrosternal pain']),
    ('DEMO-004', 'Pain radiating to left arm', 'finding', ['Left arm pain radiating from chest']),
    ('DEMO-005', 'Dyspnoea', 'finding', ['Shortness of breath', 'Breathlessness', 'Dyspnea', 'SOB']),
    ('DEMO-006', 'Palpitations', 'finding', ['Awareness of heartbeat', 'Racing heart']),
    ('DEMO-007', 'Sweating', 'finding', ['Diaphoresis', 'Excessive sweating']),
    ('DEMO-008', 'Nausea', 'finding', ['Feeling sick']),
    ('DEMO-009', 'Vomiting', 'finding', ['Being sick', 'Emesis']),
    ('DEMO-010', 'Headache', 'finding', ['Pain in head', 'Cephalgia']),
    ('DEMO-011', 'Dizziness', 'finding', ['Light-headedness', 'Giddiness']),
    ('DEMO-012', 'Fever', 'finding', ['Pyrexia', 'High temperature']),
    ('DEMO-013', 'Cough', 'finding', ['Coughing']),
    ('DEMO-014', 'Fatigue', 'finding', ['Tiredness', 'Lack of energy']),
    ('DEMO-015', 'Abdominal pain', 'finding', ['Stomach ache', 'Tummy pain']),
    ('DEMO-016', 'Back pain', 'finding', ['Backache']),
    ('DEMO-017', 'Ankle swelling', 'finding', ['Swollen ankles', 'Oedema of ankle']),
    ('DEMO-018', 'Syncope', 'finding', ['Fainting', 'Blackout']),
    ('DEMO-019', 'Weight loss', 'finding', ['Losing weight']),
    ('DEMO-020', 'Type 2 diabetes mellitus', 'disorder', ['Diabetes mellitus type 2', 'Non-insulin dependent diabetes', 'T2DM']),
    ('DEMO-021', 'Type 1 diabetes mellitus', 'disorder', ['Insulin dependent diabetes', 'T1DM']),
    ('DEMO-022', 'Diabetes mellitus', 'disorder', ['Diabetes', 'Sugar diabetes']),
    ('DEMO-023', 'Essential hypertension', 'disorder', ['High blood pressure', 'Hypertension', 'HTN']),
    ('DEMO-024', 'Hypercholesterolaemia', 'disorder', ['High cholesterol', 'Raised cholesterol']),
    ('DEMO-025', 'Asthma', 'disorder', ['Bronchial asthma']),
    ('DEMO-026', 'Chronic obstructive pulmonary disease', 'disorder', ['COPD', 'Chronic obstructive lung disease']),
    ('DEMO-027', 'Myocardial infarction', 'disorder', ['Heart attack', 'MI']),
    ('DEMO-028', 'Angina pectoris', 'disorder', ['Angina']),
    ('DEMO-029', 'Ischaemic heart disease', 'disorder', ['Coronary artery disease', 'IHD']),
    ('DEMO-030', 'Atrial fibrillation', 'disorder', ['AF', 'Irregular heartbeat']),
    ('DEMO-031', 'Heart failure', 'disorder', ['Cardiac failure']),
    ('DEMO-032', 'Cerebrovascular accident', 'disorder', ['Stroke', 'CVA']),
    ('DEMO-033', 'Chronic kidney disease', 'disorder', ['CKD', 'Chronic renal impairment']),
    ('DEMO-034', 'Hypothyroidism', 'disorder', ['Underactive thyroid']),
    ('DEMO-035', 'Depressive disorder', 'disorder', ['Depression']),
    ('DEMO-036', 'Anxiety disorder', 'disorder', ['Anxiety']),
    ('DEMO-037', 'Osteoarthritis', 'disorder', ['Degenerative joint disease', 'Wear and tear arthritis']),
    ('DEMO-038', 'Gastro-oesophageal reflux disease', 'disorder', ['GORD', 'Acid reflux', 'Heartburn']),
    ('DEMO-039', 'Malignant neoplasm of breast', 'disorder', ['Breast cancer']),
    ('DEMO-040', 'Migraine', 'disorder', ['Migraine headache']),
    ('DEMO-041', 'Appendicectomy', 'procedure', ['Appendix removal', 'Appendectomy']),
    ('DEMO-042', 'Cholecystectomy', 'procedure', ['Gallbladder removal']),
    ('DEMO-043', 'Coronary artery bypass graft', 'procedure', ['CABG', 'Heart bypass']),
    ('DEMO-044', 'Metformin', 'substance', ['Metformin hydrochloride']),
    ('DEMO-045', 'Ramipril', 'substance', []),
    ('DEMO-046', 'Atorvastatin', 'substance', ['Atorvastatin calcium']),
    ('DEMO-047', 'Aspirin', 'substance', ['Acetylsalicylic acid']),
    ('DEMO-048', 'Amlodipine', 'substance', ['Amlodipine besilate']),
    ('DEMO-049', 'Paracetamol', 'substance', ['Acetaminophen']),
    ('DEMO-050', 'Ibuprofen', 'substance', []),
    ('DEMO-051', 'Salbutamol', 'substance', ['Albuterol']),
    ('DEMO-052', 'Levothyroxine', 'substance', ['Thyroxine']),
    ('DEMO-053', 'Omeprazole', 'substance', []),
    ('DEMO-054', 'Insulin', 'substance', []),
    ('DEMO-055', 'Penicillin', 'substance', ['Penicillin antibiotic']),
    ('DEMO-056', 'Allergy to penicillin', 'finding', ['Penicillin allergy']),
    ('DEMO-057', 'Allergy to peanut', 'finding', ['Peanut allergy']),
    ('DEMO-058', 'Allergy to aspirin', 'finding', ['Aspirin allergy']),
    ('DEMO-059', 'Latex allergy', 'finding', ['Allergy to latex']),
    ('DEMO-060', 'No known allergy', 'situation', ['No known allergies', 'NKA']),
    ('DEMO-061', 'No known drug allergy', 'situation', ['NKDA', 'No known drug allergies']),
    ('DEMO-062', 'Family history of myocardial infarction', 'situation', ['FH: Myocardial infarction', 'Family history of heart attack']),
    ('DEMO-063', 'Family history of ischaemic heart disease', 'situation', ['FH: Ischaemic heart disease']),
    ('DEMO-064', 'Family history of diabetes mellitus', 'situation', ['FH: Diabetes mellitus']),
    ('DEMO-065', 'Family history of hypertension', 'situation', ['FH: Hypertension']),
    ('DEMO-066', 'Family history of cerebrovascular accident', 'situation', ['FH: Stroke']),
    ('DEMO-067', 'Family history of malignant neoplasm of breast', 'situation', ['FH: Breast cancer']),
    ('DEMO-068', 'No family history of cardiovascular disease', 'situation', []),
    ('DEMO-069', 'Smoker', 'finding', ['Current smoker', 'Cigarette smoker']),
    ('DEMO-070', 'Ex-smoker', 'finding', ['Former smoker', 'Stopped smoking']),
    ('DEMO-071', 'Never smoked tobacco', 'finding', ['Non-smoker', 'Never smoker']),
    ('DEMO-072', 'Drinks alcohol', 'finding', ['Alcohol drinker', 'Current drinker of alcohol']),
    ('DEMO-073', 'Non-drinker of alcohol', 'finding', ['Teetotaller', 'Does not drink alcohol']),
    ('DEMO-074', 'Alcohol intake above recommended limits', 'finding', ['Excessive alcohol consumption']),
    ('DEMO-075', 'Lives alone', 'finding', []),
    ('DEMO-076', 'Lives with spouse', 'finding', ['Lives with partner']),
    ('DEMO-077', 'Blood pressure', 'observable entity', ['BP']),
    ('DEMO-078', 'Heart rate', 'observable entity', ['Pulse rate']),
]
DEMO_MAPS = [
    ('DEMO-020', 'ICD-10 (demo)', 'E11', 'Type 2 diabetes mellitus'),
    ('DEMO-023', 'ICD-10 (demo)', 'I10', 'Essential (primary) hypertension'),
    ('DEMO-025', 'ICD-10 (demo)', 'J45', 'Asthma'),
    ('DEMO-001', 'ICD-10 (demo)', 'R07.4', 'Chest pain, unspecified'),
    ('DEMO-005', 'ICD-10 (demo)', 'R06.0', 'Dyspnoea'),
    ('DEMO-010', 'ICD-10 (demo)', 'R51', 'Headache'),
    ('DEMO-027', 'ICD-10 (demo)', 'I21', 'Acute myocardial infarction'),
    ('DEMO-030', 'ICD-10 (demo)', 'I48', 'Atrial fibrillation and flutter'),
    ('DEMO-056', 'ICD-10 (demo)', 'Z88.0', 'Personal history of allergy to penicillin'),
    ('DEMO-064', 'ICD-10 (demo)', 'Z83.3', 'Family history of diabetes mellitus'),
    ('DEMO-063', 'ICD-10 (demo)', 'Z82.4', 'Family history of ischaemic heart disease and other diseases of the circulatory system'),
    ('DEMO-062', 'ICD-10 (demo)', 'Z82.4', 'Family history of ischaemic heart disease and other diseases of the circulatory system'),
    ('DEMO-069', 'ICD-10 (demo)', 'Z72.0', 'Tobacco use'),
]

In [ ]:
def upsert_terminology(concept_rows, description_rows):
    # concept_rows: (id, fsn, preferred, tag, source) · description_rows: (id, concept_id, term, is_fsn)
    with conn.transaction():
        conn.execute("CREATE TEMP TABLE sc (concept_id TEXT, fsn TEXT, preferred TEXT, semantic_tag TEXT, source TEXT) ON COMMIT DROP")
        conn.execute("CREATE TEMP TABLE sd (description_id TEXT, concept_id TEXT, term TEXT, is_fsn BOOLEAN) ON COMMIT DROP")
        with conn.cursor() as cur:
            with cur.copy("COPY sc FROM STDIN") as cp:
                for r in concept_rows:
                    cp.write_row(r)
            with cur.copy("COPY sd FROM STDIN") as cp:
                for r in description_rows:
                    cp.write_row(r)
        conn.execute('''INSERT INTO concepts (concept_id, fsn, preferred, semantic_tag, source)
                        SELECT * FROM sc ON CONFLICT (concept_id) DO UPDATE SET preferred = EXCLUDED.preferred,
                        semantic_tag = EXCLUDED.semantic_tag, fsn = EXCLUDED.fsn, source = EXCLUDED.source''')
        conn.execute('''INSERT INTO descriptions (description_id, concept_id, term, is_fsn)
                        SELECT * FROM sd ON CONFLICT (description_id) DO NOTHING''')


def load_demo():
    concepts = [(cid, f"{p} ({tag})", p, tag, "demo") for cid, p, tag, syn in DEMO_CONCEPTS]
    descs = [(f"{cid}-{i}", cid, t, i == 0) for cid, p, tag, syn in DEMO_CONCEPTS for i, t in enumerate([p] + syn)]
    upsert_terminology(concepts, descs)
    with conn.cursor() as cur:
        cur.executemany("INSERT INTO code_maps (concept_id, code_system, code, code_term) VALUES (%s, %s, %s, %s) "
                        "ON CONFLICT DO NOTHING", DEMO_MAPS)
    embed_missing("SELECT description_id, term FROM descriptions WHERE embedding IS NULL AND concept_id LIKE 'DEMO-%%'")


def embed_missing(select_sql, params=()):
    rows = conn.execute(select_sql, params).fetchall()
    for i in range(0, len(rows), 100):
        part = rows[i:i + 100]
        vectors = embed_texts([t for _, t in part])
        with conn.cursor() as cur:
            cur.executemany("UPDATE descriptions SET embedding = %s::vector WHERE description_id = %s",
                            [(to_vector(v), d) for (d, _), v in zip(part, vectors)])
    return len(rows)

In [ ]:
# ---------- 4B: real SNOMED CT from NHS TRUD ----------
import csv, glob, sys, zipfile
csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

FSN, PREFERRED = "900000000000003001", "900000000000548007"
LANG_REFSETS = ["900000000000508004", "999001261000000100", "999000691000001104", "900000000000509007"]
KEEP_TAGS = ["disorder", "finding", "situation", "procedure", "substance", "observable entity"]
# Map refsets: check the IDs against the release notes of the edition you download
MAP_REFSETS = {"447562003": "ICD-10 (International map)", "999002271000000101": "ICD-10 (UK map)",
               "999002741000000101": "OPCS-4 (UK map)"}


def download_trud(api_key, item, folder="snomed_release"):
    info = requests.get(f"https://isd.digital.nhs.uk/trud/api/v1/keys/{api_key}/items/{item}/releases?latest",
                        timeout=60).json()
    release = info["releases"][0]
    zip_path = release["archiveFileName"]
    if not os.path.exists(zip_path):
        print("Downloading", zip_path, "...")
        with requests.get(release["archiveFileUrl"], stream=True, timeout=1800) as r:
            r.raise_for_status()
            with open(zip_path, "wb") as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
    zipfile.ZipFile(zip_path).extractall(folder)
    for inner in glob.glob(f"{folder}/**/*.zip", recursive=True):   # some releases contain zips inside
        zipfile.ZipFile(inner).extractall(inner[:-4])
    return folder


def rf2_rows(path):
    with open(path, encoding="utf-8", newline="") as fh:
        yield from csv.DictReader(fh, delimiter="\t", quoting=csv.QUOTE_NONE)


def snapshot_files(folder, pattern):
    return [p for p in glob.glob(f"{folder}/**/{pattern}", recursive=True) if "Full" not in p and "Delta" not in p]


def load_snomed(folder):
    active = {r["id"] for f in snapshot_files(folder, "sct2_Concept_*Snapshot*.txt") for r in rf2_rows(f) if r["active"] == "1"}
    preferred_ids = {r["referencedComponentId"] for f in snapshot_files(folder, "der2_cRefset_Language*Snapshot*.txt")
                     for r in rf2_rows(f) if r["active"] == "1" and r["acceptabilityId"] == PREFERRED
                     and r["refsetId"] in LANG_REFSETS}
    fsn, pref, syns = {}, {}, {}
    for f in snapshot_files(folder, "sct2_Description_*Snapshot*.txt"):
        for r in rf2_rows(f):
            cid = r["conceptId"]
            if r["active"] != "1" or cid not in active:
                continue
            if r["typeId"] == FSN:
                fsn[cid] = (r["id"], r["term"])
            else:
                syns.setdefault(cid, []).append((r["id"], r["term"]))
                if r["id"] in preferred_ids:
                    pref.setdefault(cid, r["term"])
    concepts, descs = [], []
    for cid, (fid, full) in fsn.items():
        tag = full[full.rfind("(") + 1:-1] if full.endswith(")") else ""
        if tag not in KEEP_TAGS:
            continue
        name = full[:full.rfind("(")].strip()
        concepts.append((cid, full, pref.get(cid, name), tag, "snomed"))
        seen = set()
        for did, term, is_fsn in [(fid, name, True)] + [(d, t, False) for d, t in syns.get(cid, [])]:
            if term.lower() not in seen:
                seen.add(term.lower())
                descs.append((did, cid, term, is_fsn))
    print(f"Writing {len(concepts):,} concepts and {len(descs):,} names to Neon ...")
    upsert_terminology(concepts, descs)

    maps = []
    for f in snapshot_files(folder, "der2_*Map*Snapshot*.txt"):
        for r in rf2_rows(f):
            system = MAP_REFSETS.get(r.get("refsetId"))
            target = (r.get("mapTarget") or "").strip()
            if system and r.get("active") == "1" and target and not target.startswith("#"):
                maps.append((r["referencedComponentId"], system, target, None,
                             int(r.get("mapGroup") or 1), int(r.get("mapPriority") or 1), r.get("mapRule")))
    with conn.cursor() as cur:
        cur.executemany("INSERT INTO code_maps VALUES (%s, %s, %s, %s, %s, %s, %s) ON CONFLICT DO NOTHING", maps)
    print(f"{len(maps):,} ICD-10 / OPCS-4 map rows")

In [ ]:
if TERMINOLOGY == "demo":
    load_demo()
elif conn.execute("SELECT count(*) FROM concepts WHERE source = 'snomed'").fetchone()[0] == 0:
    load_snomed(download_trud(TRUD_API_KEY, TRUD_ITEM))

TERMINOLOGY_SOURCE = "demo" if TERMINOLOGY == "demo" else "snomed"
print(conn.execute("SELECT source, count(*) FROM concepts GROUP BY source").fetchall())

---
# Part 5 — Clinical entity extraction

Gemini reads a conversation and returns one row per clinical fact, with a **confidence** (0–1) that we test in RQ3.

In [ ]:
SECTIONS = {
    "chief_complaint": "Chief complaint",
    "history_presenting_complaint": "History of presenting complaint",
    "past_medical_history": "Past medical history",
    "surgical_history": "Past surgical history",
    "medications": "Medications",
    "allergies": "Allergies",
    "family_history": "Family history",
    "smoking": "Smoking",
    "alcohol": "Alcohol",
    "social_history": "Social history (other)",
    "immunizations": "Immunizations",
    "gynae_obstetric_history": "Gynae / obstetric history",
    "review_of_systems": "Review of systems",
    "examination": "Examination",
    "assessment_plan": "Assessment / plan",
}

EXTRACT_PROMPT = '''Read this doctor-patient conversation and list every clinically relevant statement.

SECTIONS (use these keys exactly):
{sections}

FOR EACH STATEMENT RETURN:
- "section": one key from the list
- "text_span": the words from the conversation that support it (short)
- "coding_term": a standard clinical term to look up in SNOMED CT, e.g.
    "my chest feels tight" -> "Chest tightness"; "I've got sugar" -> "Diabetes mellitus";
    "metformin 500 twice a day" -> "Metformin" (medication = generic drug name only);
    "penicillin gives me a rash" -> "Allergy to penicillin"; "no allergies" -> "No known allergy";
    "dad had a heart attack" -> "Family history of myocardial infarction"; "quit smoking" -> "Ex-smoker";
    "had my appendix out" -> "Appendicectomy"
- "status": "present", "absent" (denied / negative), "past" (resolved) or "uncertain"
- "subject": "patient", or the relative for family history ("father", "mother", "sister" ...)
- "details": onset, duration, severity, dose, frequency, reaction, amount. "" if none.
- "confidence": 0.0-1.0, how sure you are that this statement is really in the conversation as described.

RULES: only what is said; a negative answer is still a finding ("any fever?" "no" -> "Fever", "absent");
one finding per medication, allergy and relative's condition; no names, ages, ethnicity or other identifiers.

Return JSON: {{"findings": [{{"section": "...", "text_span": "...", "coding_term": "...", "status": "...", "subject": "...", "details": "...", "confidence": 0.9}}]}}

CONVERSATION:
{dialogue}'''


def extract_findings(dialogue):
    sections = "\n".join(f"  {k}: {v}" for k, v in SECTIONS.items())
    data = ask_ai_json(EXTRACT_PROMPT.format(sections=sections, dialogue=dialogue))
    out = []
    for f in data.get("findings", []):
        if not f.get("coding_term"):
            continue
        try:
            conf = float(f.get("confidence", 0.5))
        except (TypeError, ValueError):
            conf = 0.5
        out.append({
            "section": f.get("section") if f.get("section") in SECTIONS else "assessment_plan",
            "text_span": str(f.get("text_span", "")),
            "coding_term": str(f["coding_term"]).strip(),
            "status": f.get("status") if f.get("status") in ("present", "absent", "past", "uncertain") else "present",
            "subject": str(f.get("subject") or "patient").lower(),
            "details": str(f.get("details", "")),
            "confidence": max(0.0, min(1.0, conf)),
        })
    return out

In [ ]:
SAMPLE_CONVERSATION = """
Doctor: Good morning, come in and have a seat. What's brought you in today?
Patient: Morning doctor. I've been getting this tight feeling in my chest, mostly when I walk up the hill to the shops.
Doctor: How long has that been going on?
Patient: About three weeks now. It goes off after I rest for a few minutes.
Doctor: Does the pain go anywhere else, like your arm or jaw?
Patient: Sometimes down my left arm, yes.
Doctor: Any shortness of breath, sweating or feeling sick with it?
Patient: A bit breathless, but no sweating and no sickness.
Doctor: Any palpitations or blackouts?
Patient: No, nothing like that.
Doctor: Okay. Do you have any medical conditions I should know about?
Patient: I've got type 2 diabetes, diagnosed about eight years ago, and high blood pressure.
Doctor: Any operations in the past?
Patient: I had my appendix out when I was a teenager.
Doctor: What medications are you taking at the moment?
Patient: Metformin 500 milligrams twice a day, ramipril 5 milligrams in the morning, and atorvastatin 20 at night.
Doctor: Any allergies to medicines?
Patient: Yes, penicillin. It gave me a bad rash all over.
Doctor: Does anything run in the family, any heart problems?
Patient: My dad had a heart attack when he was 55. My mum has diabetes as well.
Doctor: Do you smoke?
Patient: I used to, about 20 a day for 20 years, but I stopped five years ago.
Doctor: And alcohol?
Patient: Just a couple of pints at the weekend, maybe 8 units a week.
Doctor: Who's at home with you?
Patient: My wife.
Doctor: Thank you. I'd like to check your blood pressure and do an ECG, and I'm going to refer you to the rapid access chest pain clinic because this sounds like it could be angina.
"""

In [ ]:
demo_findings = extract_findings(SAMPLE_CONVERSATION)
pd.DataFrame(demo_findings)

---
# Part 6 — SNOMED CT coding (hybrid search + RAG)

1. **Keyword search** (PostgreSQL full text) finds names that share words with the term.
2. Any of those names without an embedding get one now (**on-demand embedding**, cached in Neon).
3. **Semantic search** (pgvector `<=>`) ranks names by meaning.
4. The two rankings are merged (Reciprocal Rank Fusion), filtered to the SNOMED hierarchy that fits the section.
5. Gemini picks one candidate or `NONE`. A code that is not in the candidate list is rejected — no invented codes.

In [ ]:
SECTION_TAGS = {
    "chief_complaint": ["finding", "disorder"],
    "history_presenting_complaint": ["finding", "disorder", "observable entity"],
    "past_medical_history": ["disorder", "finding", "situation", "procedure"],
    "surgical_history": ["procedure", "situation"],
    "medications": ["substance", "product", "medicinal product"],
    "allergies": ["finding", "disorder", "situation"],
    "family_history": ["situation", "finding"],
    "smoking": ["finding", "observable entity", "situation"],
    "alcohol": ["finding", "observable entity", "situation"],
    "social_history": ["finding", "situation", "observable entity"],
    "immunizations": ["procedure", "situation", "finding"],
    "gynae_obstetric_history": ["finding", "disorder", "procedure", "situation"],
    "review_of_systems": ["finding", "disorder"],
    "examination": ["finding", "disorder", "observable entity"],
    "assessment_plan": None,
}


def keyword_candidates(term, tags, n=40):
    tag_filter = "AND c.semantic_tag = ANY(%(tags)s)" if tags else ""
    return conn.execute(f'''
        WITH q AS (SELECT NULLIF(replace(plainto_tsquery('english', %(t)s)::text, '&', '|'), '')::tsquery AS query)
        SELECT d.description_id, d.concept_id, d.term, d.embedding IS NULL AS needs_embedding
        FROM descriptions d JOIN concepts c ON c.concept_id = d.concept_id, q
        WHERE q.query IS NOT NULL AND d.tsv @@ q.query AND c.active
          AND c.source = %(src)s {tag_filter}
        ORDER BY ts_rank_cd(d.tsv, q.query, 1) DESC, length(d.term)
        LIMIT %(n)s''', {"t": term, "tags": tags, "n": n, "src": TERMINOLOGY_SOURCE}).fetchall()


def semantic_candidates(vector, tags, n=40):
    tag_filter = "AND c.semantic_tag = ANY(%(tags)s)" if tags else ""
    try:
        conn.execute("SET hnsw.iterative_scan = relaxed_order")   # pgvector >= 0.8: keeps the tag filter accurate
    except Exception:
        pass
    return conn.execute(f'''
        SELECT d.concept_id, d.embedding <=> %(v)s::vector AS distance
        FROM descriptions d JOIN concepts c ON c.concept_id = d.concept_id
        WHERE d.embedding IS NOT NULL AND c.active AND c.source = %(src)s {tag_filter}
        ORDER BY d.embedding <=> %(v)s::vector
        LIMIT %(n)s''', {"v": vector, "tags": tags, "n": n, "src": TERMINOLOGY_SOURCE}).fetchall()


def concept_names(ids):
    rows = conn.execute("SELECT concept_id, preferred, semantic_tag FROM concepts WHERE concept_id = ANY(%s)",
                        (list(ids),)).fetchall()
    return {r[0]: (r[1], r[2]) for r in rows}


def get_maps(concept_id):
    rows = conn.execute("SELECT code_system, code FROM code_maps WHERE concept_id = %s "
                        "ORDER BY code_system, map_group, map_priority", (concept_id,)).fetchall()
    return [{"code_system": s, "code": c} for s, c in rows]


CHOOSE_PROMPT = '''You are a clinical coder. For each numbered statement choose the ONE SNOMED CT concept from
ITS candidate list that means the same thing, or "NONE".
- Only a concept_id from that statement's list. Prefer the most specific concept the statement fully supports.
- Family history -> a "Family history of ..." concept. Medication -> the drug itself.
- "confidence": 0.0-1.0 that the code is correct.
Return JSON: {{"choices": [{{"n": 1, "concept_id": "...", "confidence": 0.9}}]}}

{items}'''


def code_findings(findings):
    if not findings:
        return findings
    tags = [SECTION_TAGS.get(f["section"]) for f in findings]
    kw = [keyword_candidates(f["coding_term"], t) for f, t in zip(findings, tags)]
    missing = {row[0]: row[2] for rows in kw for row in rows if row[3]}            # on-demand embeddings
    if missing:
        ids = list(missing)
        vectors = embed_texts([missing[i] for i in ids])
        with conn.cursor() as cur:
            cur.executemany("UPDATE descriptions SET embedding = %s::vector WHERE description_id = %s",
                            [(to_vector(v), i) for i, v in zip(ids, vectors)])
    query_vectors = embed_texts([f["coding_term"] for f in findings], kind="query")

    for f, t, kw_rows, qv in zip(findings, tags, kw, query_vectors):
        scores = {}
        for ranked in ([r[1] for r in kw_rows], [r[0] for r in semantic_candidates(to_vector(qv), t)]):
            unique = list(dict.fromkeys(ranked))
            for rank, cid in enumerate(unique, 1):
                scores[cid] = scores.get(cid, 0) + 1 / (60 + rank)
        best = sorted(scores, key=scores.get, reverse=True)[:TOP_K]
        names = concept_names(best)
        f["candidates"] = [{"concept_id": c, "term": names[c][0], "semantic_tag": names[c][1]} for c in best if c in names]

    for start in range(0, len(findings), 10):
        part = findings[start:start + 10]
        items = "\n\n".join(
            f"{n}. Section: {f['section']} | Term: {f['coding_term']} | Said: \"{f['text_span']}\"\n   Candidates:\n" +
            ("\n".join(f"    {c['concept_id']} | {c['term']} ({c['semantic_tag']})" for c in f["candidates"]) or "    (none)")
            for n, f in enumerate(part, 1))
        choices = {c.get("n"): c for c in ask_ai_json(CHOOSE_PROMPT.format(items=items)).get("choices", [])}
        for n, f in enumerate(part, 1):
            c = choices.get(n, {})
            allowed = {x["concept_id"]: x for x in f["candidates"]}
            cid = str(c.get("concept_id", "")).strip()
            if cid in allowed:
                f["concept_id"], f["concept_term"] = cid, allowed[cid]["term"]
                f["code_confidence"] = float(c.get("confidence") or 0)
            else:
                f["concept_id"], f["concept_term"], f["code_confidence"] = None, None, 0.0
            f["maps"] = get_maps(f["concept_id"]) if f["concept_id"] else []
    return findings


demo_findings = code_findings(demo_findings)
pd.DataFrame(demo_findings)[["section", "coding_term", "status", "concept_id", "concept_term", "code_confidence"]]

---
# Part 7 — FHIR R4 resources

Each coded finding becomes a FHIR resource, all inside one **Bundle** per conversation:

| Section | FHIR resource |
|---|---|
| Chief complaint, history, PMH, review of systems, exam, assessment | `Condition` (absent → verificationStatus *refuted*; past → clinicalStatus *resolved*) |
| Medications | `MedicationStatement` |
| Allergies | `AllergyIntolerance` |
| Family history | `FamilyMemberHistory` (relationship + condition) |
| Smoking | `Observation` LOINC 72166-2 *Tobacco smoking status* |
| Alcohol, other social history | `Observation` (social-history category) |
| Surgical history, immunizations | `Procedure` |

Codes from real SNOMED use `http://snomed.info/sct`; demo codes use a clearly fake system so they can never be mistaken for SNOMED.
Every resource is checked against the FHIR schema with the `fhir.resources` library (R4B models).

In [ ]:
import uuid
from fhir.resources.R4B.bundle import Bundle

SNOMED = "http://snomed.info/sct"
DEMO_SYSTEM = "urn:demo:not-snomed"
TERM = "http://terminology.hl7.org/CodeSystem/"
RELATIONS = {"father": "FTH", "mother": "MTH", "sister": "SIS", "brother": "BRO", "son": "SON", "daughter": "DAU",
             "grandfather": "GRFTH", "grandmother": "GRMTH", "uncle": "UNCLE", "aunt": "AUNT", "sibling": "SIB",
             "parent": "PRN", "child": "CHILD"}


def concept(f):
    cc = {"text": f.get("concept_term") or f["coding_term"]}
    if f.get("concept_id"):
        system = DEMO_SYSTEM if str(f["concept_id"]).startswith("DEMO-") else SNOMED
        cc["coding"] = [{"system": system, "code": f["concept_id"], "display": f["concept_term"]}]
    return cc


def coding(system, code, display=None):
    c = {"system": system, "code": code}
    if display:
        c["display"] = display
    return {"coding": [c]}


def to_resource(f, patient_ref, encounter_ref):
    sec, status = f["section"], f["status"]
    note = [{"text": f"Said: {f['text_span']}"}] if f.get("text_span") else []
    if sec == "medications":
        return {"resourceType": "MedicationStatement", "status": "not-taken" if status == "absent" else "active",
                "medicationCodeableConcept": concept(f), "subject": patient_ref, "context": encounter_ref,
                **({"dosage": [{"text": f["details"]}]} if f.get("details") else {}), "note": note}
    if sec == "allergies":
        r = {"resourceType": "AllergyIntolerance", "patient": patient_ref, "encounter": encounter_ref, "code": concept(f),
             "clinicalStatus": coding(TERM + "allergyintolerance-clinical", "active"),
             "verificationStatus": coding(TERM + "allergyintolerance-verification",
                                          "refuted" if status == "absent" else "confirmed"), "note": note}
        if f.get("details") and status != "absent":
            r["reaction"] = [{"manifestation": [{"text": f["details"]}]}]
        return r
    if sec == "family_history":
        rel = RELATIONS.get(f["subject"], "FAMMEMB")
        return {"resourceType": "FamilyMemberHistory", "status": "completed", "patient": patient_ref,
                "relationship": {**coding(TERM + "v3-RoleCode", rel), "text": f["subject"]},
                "condition": [{"code": concept(f)}], "note": note}
    if sec in ("smoking", "alcohol", "social_history"):
        code_cc = ({**coding("http://loinc.org", "72166-2", "Tobacco smoking status")} if sec == "smoking"
                   else {"text": "Alcohol use" if sec == "alcohol" else "Social history"})
        return {"resourceType": "Observation", "status": "final", "subject": patient_ref, "encounter": encounter_ref,
                "category": [coding(TERM + "observation-category", "social-history")],
                "code": code_cc, "valueCodeableConcept": concept(f), "note": note}
    if sec in ("surgical_history", "immunizations"):
        return {"resourceType": "Procedure", "status": "not-done" if status == "absent" else "completed",
                "subject": patient_ref, "encounter": encounter_ref, "code": concept(f), "note": note}
    category = "problem-list-item" if sec == "past_medical_history" else "encounter-diagnosis"
    r = {"resourceType": "Condition", "subject": patient_ref, "encounter": encounter_ref, "code": concept(f),
         "category": [coding(TERM + "condition-category", category)], "note": note}
    if status == "absent":
        r["verificationStatus"] = coding(TERM + "condition-ver-status", "refuted")   # no clinicalStatus allowed then
    else:
        r["verificationStatus"] = coding(TERM + "condition-ver-status", "provisional" if status == "uncertain" else "confirmed")
        r["clinicalStatus"] = coding(TERM + "condition-clinical", "resolved" if status == "past" else "active")
    return r


def make_bundle(item_id, findings):
    ns = uuid.uuid5(uuid.NAMESPACE_URL, f"conversation/{item_id}")
    pid, eid = f"urn:uuid:{uuid.uuid5(ns, 'patient')}", f"urn:uuid:{uuid.uuid5(ns, 'encounter')}"
    entries = [
        {"fullUrl": pid, "resource": {"resourceType": "Patient", "identifier": [{"system": "urn:study:id", "value": str(item_id)}]}},
        {"fullUrl": eid, "resource": {"resourceType": "Encounter", "status": "finished", "subject": {"reference": pid},
                                      "class": {"system": TERM + "v3-ActCode", "code": "AMB", "display": "ambulatory"}}},
    ]
    for i, f in enumerate(findings):
        res = to_resource(f, {"reference": pid}, {"reference": eid})
        if not res.get("note"):
            res.pop("note", None)
        entries.append({"fullUrl": f"urn:uuid:{uuid.uuid5(ns, str(i))}", "resource": res})
    return {"resourceType": "Bundle", "type": "collection", "entry": entries}


def validate_bundle(bundle):
    # returns (number of valid resources, total, list of errors)
    from importlib import import_module
    ok, errors = 0, []
    for e in bundle["entry"]:
        rtype = e["resource"]["resourceType"]
        model = getattr(import_module(f"fhir.resources.R4B.{rtype.lower()}"), rtype)
        try:
            model.model_validate(e["resource"])
            ok += 1
        except Exception as ex:
            errors.append(f"{rtype}: {str(ex).splitlines()[0][:120]}")
    if not errors:
        Bundle.model_validate(bundle)
    return ok, len(bundle["entry"]), errors


demo_bundle = make_bundle("sample", demo_findings)
print("Valid FHIR resources:", validate_bundle(demo_bundle)[:2])
print(json.dumps(demo_bundle["entry"][2]["resource"], indent=2)[:900])

---
# Part 8 — Experiment on MTS-Dialog

## 8.1 Load the data and pick the conversations
We use the **official test set 1** (200 conversations, MEDIQA-Chat 2023). Each conversation comes with the
**section header** and the **section text** an expert wrote from it — our gold standard. We keep the
history sections and take `PER_SECTION` conversations from each (fixed random seed, so results are repeatable).

In [ ]:
MTS_URL = ("https://raw.githubusercontent.com/abachaa/MTS-Dialog/main/Main-Dataset/"
           "MTS-Dialog-TestSet-1-MEDIQA-Chat-2023.csv")
mts = pd.read_csv(MTS_URL)
print(len(mts), "conversations in test set 1")

parts = []
for header in TEST_SECTIONS:
    group = mts[mts.section_header == header]
    parts.append(group if PER_SECTION is None else group.sample(min(len(group), PER_SECTION), random_state=RANDOM_SEED))
sample = pd.concat(parts).reset_index(drop=True)
sample["item_id"] = "mts-test1-" + sample["ID"].astype(str)
print(len(sample), "conversations selected")
sample.section_header.value_counts()

## 8.2 Run the pipeline on every conversation
Results are saved in Neon after each conversation, so if Colab disconnects or Gemini runs out of free quota,
**just run this cell again** — finished conversations are skipped.

In [ ]:
SETTINGS = {"text_model": TEXT_MODEL, "embedding_model": EMBEDDING_MODEL, "terminology": TERMINOLOGY}
done = {r[0] for r in conn.execute("SELECT item_id FROM results WHERE source = 'mts' AND fhir IS NOT NULL").fetchall()}

for i, row in sample.iterrows():
    if row.item_id in done:
        continue
    print(f"[{i + 1}/{len(sample)}] {row.item_id} ({row.section_header})")
    findings = code_findings(extract_findings(row.dialogue))
    bundle = make_bundle(row.item_id, findings)
    conn.execute('''INSERT INTO results (item_id, source, section_header, dialogue, gold, findings, fhir, settings)
                    VALUES (%s, 'mts', %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (item_id) DO UPDATE SET findings = EXCLUDED.findings, fhir = EXCLUDED.fhir,
                    judge = NULL, settings = EXCLUDED.settings''',
                 (row.item_id, row.section_header, row.dialogue, row.section_text,
                  json.dumps(findings), json.dumps(bundle), json.dumps(SETTINGS)))
print("Pipeline finished")

## 8.3 Evaluation judge
For each conversation a separate Gemini call (the **judge**) does two checks, following the fact-based evaluation
used in the MTS-Dialog paper:
- **Recall:** it splits the expert note into atomic clinical facts and marks which ones our findings capture.
- **Precision:** it marks each of our findings as *supported* or *not supported* by the conversation
  (not supported = hallucination).

Demographics (age, sex, ethnicity) are ignored because the extractor is told not to output identifiers.

In [ ]:
JUDGE_PROMPT = '''You are evaluating an automatic clinical information extraction system.

CONVERSATION:
{dialogue}

EXPERT NOTE (gold standard):
{gold}

SYSTEM FINDINGS (numbered):
{findings}

TASK 1: Split the EXPERT NOTE into atomic clinical facts (one symptom, condition, medication, allergy, relative's
condition, habit, negative finding, etc. each). IGNORE age, sex, ethnicity and names. For each fact say whether
the SYSTEM FINDINGS capture it (same meaning, including whether it is present or absent).
TASK 2: For each numbered SYSTEM FINDING say whether the CONVERSATION supports it (correct meaning AND correct
present/absent status).

Return JSON: {{"gold_facts": [{{"fact": "...", "captured": true}}], "findings": [{{"n": 1, "supported": true}}]}}'''


def judge(row):
    findings = row["findings"] or []
    listing = "\n".join(f"{n}. [{f['section']}] {f['coding_term']} ({f['status']}, {f['subject']}) {f['details']}"
                        for n, f in enumerate(findings, 1)) or "(none)"
    return ask_ai_json(JUDGE_PROMPT.format(dialogue=row["dialogue"], gold=row["gold"], findings=listing), model=JUDGE_MODEL)


todo = conn.execute("SELECT item_id, dialogue, gold, findings FROM results WHERE source = 'mts' AND judge IS NULL").fetchall()
for k, (item_id, dialogue, gold, findings) in enumerate(todo, 1):
    print(f"judging {k}/{len(todo)} {item_id}")
    verdict = judge({"dialogue": dialogue, "gold": gold, "findings": findings})
    conn.execute("UPDATE results SET judge = %s WHERE item_id = %s", (json.dumps(verdict), item_id))
print("Judging finished")

## 8.4 Results

In [ ]:
rows = conn.execute('''SELECT item_id, section_header, findings, fhir, judge FROM results
                       WHERE source = 'mts' AND judge IS NOT NULL AND item_id = ANY(%s)''',
                    (list(sample.item_id),)).fetchall()

# which of our sections count as "the same section" as each MTS-Dialog header
SECTION_MAP = {
    "CC": {"chief_complaint", "history_presenting_complaint"},
    "GENHX": {"history_presenting_complaint", "chief_complaint"},
    "PASTMEDICALHX": {"past_medical_history"},
    "PASTSURGICAL": {"surgical_history"},
    "MEDICATIONS": {"medications"},
    "ALLERGY": {"allergies"},
    "FAM/SOCHX": {"family_history", "smoking", "alcohol", "social_history"},
    "ROS": {"review_of_systems", "history_presenting_complaint"},
}
HEADER_NAMES = {"CC": "Chief complaint", "GENHX": "History of present illness", "PASTMEDICALHX": "Past medical history",
                "PASTSURGICAL": "Past surgical history", "MEDICATIONS": "Medications", "ALLERGY": "Allergies",
                "FAM/SOCHX": "Family / social history", "ROS": "Review of systems"}

per_conv, per_finding = [], []
for item_id, header, findings, fhir_bundle, verdict in rows:
    findings = findings or []
    gold = verdict.get("gold_facts", [])
    supported = {int(x.get("n", 0)): bool(x.get("supported")) for x in verdict.get("findings", [])}
    valid, total, errs = validate_bundle(fhir_bundle)
    per_conv.append({
        "item_id": item_id, "section": header,
        "gold_facts": len(gold), "captured": sum(bool(g.get("captured")) for g in gold),
        "findings": len(findings), "supported": sum(supported.get(n, False) for n in range(1, len(findings) + 1)),
        "section_found": any(f["section"] in SECTION_MAP.get(header, set()) for f in findings),
        "fhir_valid": valid, "fhir_total": total,
    })
    for n, f in enumerate(findings, 1):
        per_finding.append({**{k: f.get(k) for k in ("section", "coding_term", "status", "subject", "details",
                                                       "text_span", "confidence", "concept_id", "concept_term",
                                                       "code_confidence")},
                            "item_id": item_id, "gold_section": header, "supported": supported.get(n, False),
                            "codes_other": "; ".join(f"{m['code_system']}: {m['code']}" for m in f.get("maps", []))})

conv = pd.DataFrame(per_conv)
fnd = pd.DataFrame(per_finding)


def prf(d):
    p = d.supported.sum() / d.findings.sum() if d.findings.sum() else float("nan")
    r = d.captured.sum() / d.gold_facts.sum() if d.gold_facts.sum() else float("nan")
    return p, r, (2 * p * r / (p + r) if p + r else float("nan"))


p, r, f1 = prf(conv)
summary = pd.DataFrame([
    ["Conversations evaluated", len(conv)],
    ["Findings extracted", int(conv.findings.sum())],
    ["Gold facts in expert notes", int(conv.gold_facts.sum())],
    ["RQ1 · Precision (findings supported by the conversation)", f"{p:.1%}"],
    ["RQ1 · Recall (expert-note facts captured)", f"{r:.1%}"],
    ["RQ1 · F1", f"{f1:.1%}"],
    ["RQ1 · Hallucination rate (1 − precision)", f"{1 - p:.1%}"],
    ["RQ1 · Section detected correctly", f"{conv.section_found.mean():.1%}"],
    ["RQ2 · Findings given a code", f"{fnd.concept_id.notna().mean():.1%}" if len(fnd) else "-"],
    ["RQ2 · Mean coding confidence (coded findings)", f"{fnd.code_confidence[fnd.concept_id.notna()].mean():.2f}" if len(fnd) else "-"],
    ["FHIR · Structurally valid resources", f"{conv.fhir_valid.sum() / conv.fhir_total.sum():.1%}"],
    ["Terminology used", TERMINOLOGY],
    ["Models", f"{TEXT_MODEL} (judge: {JUDGE_MODEL})"],
], columns=["Measure", "Result"])
summary

### RQ4 — Results per history section

In [ ]:
by_section = []
for header in TEST_SECTIONS:
    d = conv[conv.section == header]
    if len(d):
        sp, sr, sf = prf(d)
        by_section.append({"Section": HEADER_NAMES[header], "Conversations": len(d), "Precision": sp,
                           "Recall": sr, "F1": sf, "Section detected": d.section_found.mean()})
by_section = pd.DataFrame(by_section)
by_section.style.format({c: "{:.0%}" for c in ["Precision", "Recall", "F1", "Section detected"]})

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": GRID, "axes.labelcolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})


def tidy(ax):
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(by_section))
ax.bar(x - 0.19, by_section.Precision, 0.36, color=BLUE, label="Precision (supported by conversation)")
ax.bar(x + 0.19, by_section.Recall, 0.36, color=ORANGE, label="Recall (expert-note facts captured)")
ax.set_xticks(x, by_section.Section, rotation=25, ha="right")
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title(f"Extraction accuracy by history section (overall F1 {f1:.0%})", loc="left", color=INK, fontsize=12, pad=28)
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, fontsize=9)
tidy(ax)
plt.tight_layout()
plt.savefig("fig1_accuracy_by_section.png", dpi=200)
plt.show()

### RQ3 — Does the model's own confidence predict hallucinations?
If low-confidence findings are wrong more often, a **confidence threshold** removes hallucinations at a small cost in recall.

In [ ]:
bins = [0, 0.5, 0.7, 0.9, 1.0001]
labels = ["< 0.5", "0.5–0.7", "0.7–0.9", "≥ 0.9"]
fnd["confidence_band"] = pd.cut(fnd.confidence, bins=bins, labels=labels, right=False)
calib = fnd.groupby("confidence_band", observed=False).agg(findings=("supported", "size"),
                                                           supported=("supported", "mean")).reset_index()
calib["hallucination_rate"] = 1 - calib.supported

kept = fnd[fnd.confidence >= CONFIDENCE_THRESHOLD]
dropped = len(fnd) - len(kept)
print(f"Without threshold: hallucination rate {1 - fnd.supported.mean():.1%} over {len(fnd)} findings")
print(f"With confidence ≥ {CONFIDENCE_THRESHOLD}: hallucination rate {1 - kept.supported.mean():.1%} "
      f"over {len(kept)} findings ({dropped} removed, of which {(~fnd[fnd.confidence < CONFIDENCE_THRESHOLD].supported).sum()} were hallucinations)")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar(calib.confidence_band.astype(str), calib.hallucination_rate.fillna(0), 0.55, color=BLUE)
for i, (h, n) in enumerate(zip(calib.hallucination_rate, calib.findings)):
    ax.text(i, (0 if pd.isna(h) else h) + 0.02, f"n={n}", ha="center", color=MUTED, fontsize=9)
ax.set_ylim(0, max(0.3, float(calib.hallucination_rate.fillna(0).max()) + 0.1))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Model's confidence in the finding")
ax.set_title("Hallucination rate by model confidence", loc="left", color=INK, fontsize=12)
tidy(ax)
plt.tight_layout()
plt.savefig("fig2_hallucination_by_confidence.png", dpi=200)
plt.show()
calib

### RQ2 — Coding: what got coded, and a sheet for a clinician to check
Automatic coding accuracy needs a human reference: the sheet below lists every coded finding with an empty
**Clinician: code correct? (Y/N)** column. A clinician fills it in, and the share of *Y* is the coding accuracy.

In [ ]:
coded = fnd[fnd.concept_id.notna()]
print(f"{len(coded)} of {len(fnd)} findings coded ({len(coded) / max(len(fnd), 1):.0%}) with the '{TERMINOLOGY}' terminology")
review = fnd[["item_id", "gold_section", "section", "text_span", "coding_term", "status", "subject",
              "concept_id", "concept_term", "code_confidence", "codes_other", "supported"]].copy()
review["Clinician: finding correct? (Y/N)"] = ""
review["Clinician: code correct? (Y/N)"] = ""
review["Clinician: better code"] = ""
review.head(15)

## 8.5 Download everything for your supervisor
`results_for_supervisor.zip` contains the summary, per-section results, the clinician review sheet, both charts and
all FHIR bundles.

In [ ]:
with pd.ExcelWriter("results.xlsx") as xw:
    summary.to_excel(xw, sheet_name="Summary", index=False)
    by_section.to_excel(xw, sheet_name="By section", index=False)
    calib.to_excel(xw, sheet_name="Confidence (RQ3)", index=False)
    review.to_excel(xw, sheet_name="Clinician review (RQ2)", index=False)
    conv.to_excel(xw, sheet_name="Per conversation", index=False)

bundles = {item_id: b for item_id, _, _, b, _ in rows}
with open("fhir_bundles.json", "w") as fh:
    json.dump(bundles, fh, indent=1)

with zipfile.ZipFile("results_for_supervisor.zip", "w") as z:
    for f in ["results.xlsx", "fhir_bundles.json", "fig1_accuracy_by_section.png", "fig2_hallucination_by_confidence.png"]:
        z.write(f)
print("Saved results_for_supervisor.zip")

try:
    from google.colab import files
    files.download("results_for_supervisor.zip")
except Exception:
    pass

---
# Part 9 — Your own consultation (typed or audio)

**Typed:** edit `MY_CONVERSATION` and run. **Audio:** upload a recording (role-play, with consent) — Whisper
transcribes it in Colab (use **Runtime → Change runtime type → T4 GPU** for speed); Gemini then labels Doctor/Patient.

In [ ]:
MY_CONVERSATION = '''
Doctor: What brings you in today?
Patient: I've had a cough and fever for four days.
Doctor: Any medical conditions?
Patient: Asthma. I use a salbutamol inhaler.
Doctor: Do you smoke?
Patient: Never.
'''

findings = code_findings(extract_findings(MY_CONVERSATION))
bundle = make_bundle("my-consultation", findings)
print("FHIR valid:", validate_bundle(bundle)[:2])
covered = {f["section"] for f in findings}
print("❌ Not discussed:", ", ".join(SECTIONS[s] for s in ["chief_complaint", "past_medical_history", "medications",
      "allergies", "family_history", "smoking", "alcohol"] if s not in covered) or "nothing — all core sections covered")
pd.DataFrame(findings)[["section", "coding_term", "status", "subject", "details", "confidence", "concept_term"]]

In [ ]:
# ---- audio (optional): uncomment to use ----
# !pip install -q faster-whisper
# from google.colab import files
# audio_name = list(files.upload().keys())[0]
# from faster_whisper import WhisperModel
# import torch
# whisper = WhisperModel("small", device="cuda" if torch.cuda.is_available() else "cpu",
#                        compute_type="float16" if torch.cuda.is_available() else "int8")
# segments, _ = whisper.transcribe(audio_name, language="en", vad_filter=True)
# lines = [s.text.strip() for s in segments if s.text.strip()]
# labels = ask_ai_json("Label each numbered line of this consultation as Doctor or Patient. Return JSON "
#                      '{"speakers": [...]}' + f" with {len(lines)} items.\n" +
#                      "\n".join(f"{i + 1}. {t}" for i, t in enumerate(lines)))["speakers"]
# MY_CONVERSATION = "\n".join(f"{s}: {t}" for s, t in zip(labels, lines))
# print(MY_CONVERSATION)   # then run the cell above again

---
**Citation for the data:** Ben Abacha A, Yim W, Fan Y, Lin T. *An Empirical Study of Clinical Note Generation from
Doctor-Patient Encounters.* EACL 2023, pp. 2291–2302. MTS-Dialog is licensed CC BY 4.0.

**Limitations to state when presenting:** the judge is the same model family as the extractor (a clinician check of
a sample is needed); MTS-Dialog conversations are short, single-section snippets; demo codes are not SNOMED.

In [ ]:
conn.close()